In [ ]:
# !pip install jiwer
# !pip install faster-whisper # whisper library
# !pip install deepgram-sdk # deepgram library
# !pip install httpx
# !pip install torchaudio
# !pip install librosa
# !pip install datasets
!pip install --upgrade datasets #-> datasets update if there's an error

In [ ]:
"""
Import section
"""

from transformers import (
    Wav2Vec2Processor, # sound recognision processors
    Wav2Vec2ForCTC,
    Trainer, # easy training pipeline
)

import torch


In [ ]:
############################
# Whisper dependencies
from faster_whisper import WhisperModel
############################

# Getting file from colab
# from google.colab import files
# uploaded = files.upload()
# filename = next(iter(uploaded))

###############################
# DeepGram dependencies
import requests
import wave
import io
import time
import os
import json
import threading
from datetime import datetime
from deepgram import (
  DeepgramClient,
  DeepgramClientOptions,
  AgentWebSocketEvents,
  AgentKeepAlive,
  FileSource
)
################################

# Initialize the client
# deepgram = DeepgramClient("2009ec81ca903b0b136a9427be25218e8bd65817") # write env key here

"""
MODELS
  Whisper:
    1. tiny -> ~ 39 millions
    2. medium -> ~ 769 millions
    3. large-v1 -> ~1.55 billions
    4. turbo -> ~1.55 billions, optimized version of large-v1
    5. distill-large-v3 -> ~400-500 millions, lighter than large-v3
    6. large-v3 ->  ~ 1.6 billion parameters
  Deepgram:
    1. nova2 -> official and only model which support greek language by default
  Microsoft:
    1. microsoft/wavlm-large -> ~ 94.7 million parameters


OTHER:
1. jonatasgrosman/wav2vec2-large-xlsr-53-greek -> facebook wav2vec2 finetuned greek model, ~ 317 billion parameters, 166k downloads
  Note: the only useful custom hf model, others are 1k downloads on avg, not serious
"""

## ElevenLabs can't be used, cause it has text-to-speech format only

# WHISPER Models
# whisper_tiny = WhisperModel("tiny")
# whisper_medium = WhisperModel("medium")
# whisper_large_v1 = WhisperModel("large-v1")
# whisper_turbo = WhisperModel("turbo")
# whisper_distill_large_v3 = WhisperModel("distill-large-v3")
# whisper_large_v3 = WhisperModel("large-v3")


# DEEPGRAM models
# deepgram_source = { "buffer": open(filename, "rb"), "mimetype": "audio/mp3" }
# deepgram_model = deepgram.listen.prerecorded(
#     FileSource(source=deepgram_source),
#     {"model": "nova-2", "language": "el"}  # el = Greek
# )

# Microsoft model todo


###################################
# Mozilla Common Voice for testing
from datasets import load_dataset, Audio

# Greek dataset
dataset = load_dataset("mozilla-foundation/common_voice_17_0", "el", split="train")

# Squeezing to 16hz
dataset = dataset.cast_column("audio", Audio(sampling_rate=16000))
###################################


In [ ]:
"""
Whisper Loader Pipeline
"""

class WhisperModelPipeline:
  def __init__(self, model_name, audio):
    self.model_name = model_name
    self.audio = audio
    self.model = None

  def load_model(self):
    self.model = WhisperModel(self.model_name)

  def process_logic(self):
    text_stored = ""
    segments, info = self.model.transcribe(self.audio, language="el") # <- specify language output
    for segment in segments:
      # print("[%.2fs -> %.2fs] %s" % (segment.start, segment.end, segment.text)) # <- to see with seconds
      text_stored += segment.text + " "
    return text_stored

In [ ]:
"""
  Jiwer Loader Pipeline
"""
import jiwer
from jiwer import wer,cer

"""
wer - word error rate
cer - character error rate
"""

"""

ValueError: After applying the transformation, each reference should be a list of strings, with each string being a single word or character.Please ensure the given transformation reduces the input to a list of list strings.

To fix:

jiwer.ToLowerCase(),
jiwer.RemovePunctuation(),
jiwer.RemoveMultipleSpaces(),
jiwer.Strip()

and adding a normalization
"""

class JiwerMetricsPipeline:
  def __init__(self, hypothesis, truth):
    self.hypothesis = hypothesis
    self.truth = truth

  def compute_metrics(self):
    transformation = jiwer.Compose([
        jiwer.ToLowerCase(),
        jiwer.RemovePunctuation(),
        jiwer.RemoveMultipleSpaces(),
        jiwer.Strip()
    ])

    normalized_truth = transformation(self.truth)
    normalized_hypothesis = transformation(self.hypothesis)

    wer_score = wer(normalized_truth, normalized_hypothesis)
    cer_score = cer(normalized_truth, normalized_hypothesis)

    return wer_score, cer_score

In [ ]:
# Outputs
dataset_sentence = dataset[0]["sentence"]
dataset_audio = dataset[0]["audio"]["path"]

stored_metrics = []

In [ ]:
def launch_whisper(model, stored):
  whisper = WhisperModelPipeline(model, dataset_audio)
  whisper.load_model()
  whisper_output = whisper.process_logic()

  err_metrics = JiwerMetricsPipeline(whisper_output, dataset_sentence).compute_metrics()

  stored.append({
    "model": model,
    "wer": err_metrics[0],
    "cer": err_metrics[1]
  })

  return stored

launch_whisper("tiny", stored_metrics)
stored_metrics
# launch_whisper("large-v3", stored_metrics)
# stored_metrics

